In [11]:
import rasterio
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
# defining supported formats
SUPPORTED_FORMATS = {
    ".tif",
    ".tiff",
    ".img",
    ".jp2",
    ".vrt"
}

In [14]:
# automaticaly find datasets
input_folder = Path.cwd() / "dataset"
output_folder = Path.cwd() / "reports"

output_folder.mkdir(
    exist_ok=True
)

# Search dataset folder AND all child folders
datasets = [
    file
    for file in input_folder.rglob("*")
    if file.is_file()
    and file.suffix.lower() in SUPPORTED_FORMATS
]

print(f"Found {len(datasets)} raster datasets:\n")

for dataset in datasets:
    print(dataset)

Found 1 raster datasets:

D:\imma\Geospatial\dataset\ken_ppp_2020_UNadj_constrained.tif


In [16]:
# data profiler
# limiting sample size to 100_000 
def profile_raster_data(filepath, sample_size=100_000):

    filepath = Path(filepath)

    with rasterio.open(filepath) as src:

        # 1. DATASET SUMMARY
        dataset_summary = {
            "File": filepath.name,
            "Driver": src.driver,
            "Width": src.width,
            "Height": src.height,
            "Bands": src.count,
            "CRS": str(src.crs),
            "Resolution X": src.res[0],
            "Resolution Y": src.res[1],
            "Data Types": src.dtypes,
            "NoData Values": src.nodatavals,
            "Bounds": {
                "min_x": src.bounds.left,
                "min_y": src.bounds.bottom,
                "max_x": src.bounds.right,
                "max_y": src.bounds.top
            }
        }

        # 2. BAND PROFILE
        band_profile = []

        for band_number in range(1, src.count + 1):

            # Statistics accumulated across blocks
            count = 0
            nodata_count = 0
            zero_count = 0
            negative_count = 0

            minimum = None
            maximum = None

            value_sum = 0.0
            value_sum_squared = 0.0

            # Sample used for median
            sample = []
            values_seen = 0

            rng = np.random.default_rng(42)

            # Processing raster block by block
            for _, window in src.block_windows(band_number):

                data = src.read(
                    band_number,
                    window=window,
                    masked=True
                )

                values = data.compressed()

                if len(values) == 0:
                    nodata_count += data.size
                    continue

                # Basic counts
                valid_count = len(values)

                count += valid_count
                nodata_count += data.size - valid_count

                zero_count += np.count_nonzero(values == 0)
                negative_count += np.count_nonzero(values < 0)


                # Min / Max
                block_min = values.min()
                block_max = values.max()

                if minimum is None or block_min < minimum:
                    minimum = block_min

                if maximum is None or block_max > maximum:
                    maximum = block_max

                # Mean / Std components
                values_float = values.astype(np.float64)

                value_sum += values_float.sum()
                value_sum_squared += np.square(
                    values_float
                ).sum()

                # Sample for median
                for value in values:

                    values_seen += 1

                    if len(sample) < sample_size:

                        sample.append(value)

                    else:

                        index = rng.integers(
                            0,
                            values_seen
                        )

                        if index < sample_size:
                            sample[index] = value

            # Final statistics
            if count > 0:

                mean = value_sum / count

                variance = (
                    value_sum_squared / count
                    - mean ** 2
                )

                # Protect against tiny floating-point negatives
                variance = max(variance, 0)

                std = np.sqrt(variance)

                median = np.median(sample)

            else:

                mean = np.nan
                std = np.nan
                median = np.nan


            total_pixels = src.width * src.height

            nodata_pct = (
                nodata_count / total_pixels * 100
                if total_pixels > 0
                else 0
            )

            # Band metadata
            band_profile.append({

                "Band": band_number,

                "Description":
                    src.descriptions[band_number - 1],

                "Data Type":
                    src.dtypes[band_number - 1],

                "NoData":
                    src.nodatavals[band_number - 1],

                "Min":
                    minimum,

                "Max":
                    maximum,

                "Mean":
                    mean,

                "Median":
                    median,

                "Std":
                    std,

                "Valid Pixels":
                    count,

                "NoData Count":
                    nodata_count,

                "NoData %":
                    round(nodata_pct, 2),

                "Zeros":
                    zero_count,

                "Negative":
                    negative_count
            })


        band_profile = pd.DataFrame(
            band_profile
        )

        # 3. SPATIAL / RASTER SUMMARY
        spatial_summary = {

            "CRS": str(src.crs),

            "Width": src.width,

            "Height": src.height,

            "Bands": src.count,

            "Pixel Width": src.res[0],

            "Pixel Height": src.res[1],

            "Min X": src.bounds.left,

            "Min Y": src.bounds.bottom,

            "Max X": src.bounds.right,

            "Max Y": src.bounds.top,

            "Transform": str(src.transform),

            "Tiled": src.is_tiled,

            "Block Shapes": src.block_shapes,

            "Compression":
                str(src.compression),

            "Overviews":
                {
                    band: src.overviews(band)
                    for band in range(1, src.count + 1)
                }
        }

        # 4. REPORT
        report = {

            "dataset":
                dataset_summary,

            "bands":
                band_profile,

            "spatial":
                spatial_summary
        }

        return report

In [17]:
# write report to excel workbook
def export_raster_report(report, output_path):

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl"
    ) as writer:
       
        # 1. Dataset Summary  
        dataset_summary = pd.DataFrame(
            list(report["dataset"].items()),
            columns=["Metric", "Value"]
        )

        dataset_summary.to_excel(
            writer,
            sheet_name="Dataset_Summary",
            index=False
        )

        # 2. Band Profile
        report["bands"].to_excel(
            writer,
            sheet_name="Band_Profile",
            index=False
        )

        # 3. Spatial Summary
        spatial_summary = pd.DataFrame(
            list(report["spatial"].items()),
            columns=["Metric", "Value"]
        )

        spatial_summary.to_excel(
            writer,
            sheet_name="Spatial_Summary",
            index=False
        )

In [18]:
# Export file and handle any errors
for filepath in datasets:

    print(f"\nProfiling: {filepath}")

    try:

        # Profile raster
        report = profile_raster_data(filepath)

        # Create output filename
        output_path = (
            output_folder
            / f"{filepath.stem}_rasterprofile.xlsx"
        )

        # Export report
        export_raster_report(
            report,
            output_path
        )

        print(
            f"✓ Complete: {output_path.name}"
        )

    except Exception as e:

        print(
            f"✗ Failed: {filepath.name}"
        )

        print(
            f"  Error: {e}"
        )


Profiling: D:\imma\Geospatial\dataset\ken_ppp_2020_UNadj_constrained.tif
✓ Complete: ken_ppp_2020_UNadj_constrained_rasterprofile.xlsx
